# Tree-Aware Heuristic Diagnostics

This notebook measures the surrogate final error and runtime of the tree-aware heuristic using beam width B=3. Each run uses the same benchmark cases, sensor budgets, greedy initialization, stopping threshold, iteration limit, and d-sensor move set.


In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Resolve paths whether the notebook is launched from the repository root or this folder.
working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "min-error-sequential"
    if (working_directory / "min-error-sequential").is_dir()
    else working_directory
)
MODULE_DIRECTORY = PROJECT_DIRECTORY
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import beam_width_analysis
beam_width_analysis = importlib.reload(beam_width_analysis)
beam_width_summary = beam_width_analysis.beam_width_summary
load_beam_width_results = beam_width_analysis.load_beam_width_results
make_beam_width_plot = beam_width_analysis.make_beam_width_plot
run_beam_width_study = beam_width_analysis.run_beam_width_study

print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)


## Run Settings

Set `RUN_SCOPE = "test"` for the small representative diagnostic sample or `RUN_SCOPE = "full"` for every generated expanded case. The test scope samples five cases from each selected $K,M$ cell; the full scope uses all generated cases.


In [ ]:
# Compare the expanded one-dimensional and two-dimensional benchmarks.
DATASETS = ["benchmark_1d", "benchmark_2d"]

# Switch here: "test" runs a small representative sample; "full" runs every generated case.
RUN_SCOPE = "test"
if RUN_SCOPE not in {"test", "full"}:
    raise ValueError('RUN_SCOPE must be "test" or "full"')

REPLICATES_PER_CELL = None
TEST_K_VALUES = None if RUN_SCOPE == "full" else tuple(range(3, 8))
TEST_M_VALUES = None if RUN_SCOPE == "full" else tuple(range(3, 9))
RANDOM_CASES_PER_K_M = None if RUN_SCOPE == "full" else 5
CASE_SAMPLE_SEED = 20260813

# Keep this study separate from the main algorithm comparison.
RESULTS_ROOT = MODULE_DIRECTORY / "results" / "beam_width"

print("Run scope:", RUN_SCOPE)
print("Selected K values:", TEST_K_VALUES or "all")
print("Selected M values:", TEST_M_VALUES or "all")
print("Random cases per K,M cell:", RANDOM_CASES_PER_K_M or "all")
print("Case-sampling seed:", CASE_SAMPLE_SEED)
print("Results directory:", RESULTS_ROOT)


## Heuristic Configuration

Only `BEAM_WIDTHS` changes between diagnostic runs. All beam widths start from the same Greedy allocation and use the same local-search settings, so the table and plot isolate the effect of decision-tree beam width.


In [ ]:
# Number of partial decision trees retained after each beam expansion.
BEAM_WIDTHS = (3,)

# Shared allocation-search settings for every beam width.
MAX_ITERATIONS = 5
IMPROVEMENT_THRESHOLD = 1e-5
INITIALIZATION = "greedy"
RANDOM_SEED = 0
MOVE_SIZES = (1, 2, 3)
PROGRESS_EVERY = 25

print("Beam widths:", BEAM_WIDTHS)
print("Maximum iterations:", MAX_ITERATIONS)
print("Improvement threshold:", IMPROVEMENT_THRESHOLD)
print("Initialization:", INITIALIZATION)
print("Move sizes:", MOVE_SIZES)
print("Progress interval:", PROGRESS_EVERY, "cases")


In [ ]:
records_by_dataset = {}
for dataset in DATASETS:
    display(Markdown(f"### Running `{dataset}`"))
    records_by_dataset[dataset] = run_beam_width_study(
        dataset,
        beam_widths=BEAM_WIDTHS,
        replicates_per_cell=REPLICATES_PER_CELL,
        results_root=RESULTS_ROOT,
        max_iterations=MAX_ITERATIONS,
        improvement_threshold=IMPROVEMENT_THRESHOLD,
        initialization=INITIALIZATION,
        random_seed=RANDOM_SEED,
        move_sizes=MOVE_SIZES,
        progress_every=PROGRESS_EVERY,
        k_values=TEST_K_VALUES,
        m_values=TEST_M_VALUES,
        random_cases_per_k_m=RANDOM_CASES_PER_K_M,
        case_sample_seed=CASE_SAMPLE_SEED,
    )


## Numerical Summary

This table contains the numerical values behind the beam-width diagnostic plot. The error columns are surrogate final errors from the tree-aware heuristic, averaged over the selected cases and budgets.


In [ ]:
summary_lines = [
    "| Dataset | Beam width | Runs | Mean surrogate final error | Maximum surrogate final error | Mean runtime (s) |",
    "|---|---:|---:|---:|---:|---:|",
]
for dataset, records in records_by_dataset.items():
    for row in beam_width_summary(records):
        summary_lines.append(
            f"| {dataset} | {row['beam_width']} | {row['runs']} | "
            f"{row['mean_final_error']:.5f} | "
            f"{row['maximum_final_error']:.5f} | "
            f"{row['mean_seconds']:.5f} |"
        )

display(Markdown("\n".join(summary_lines)))


## Mean Surrogate Final Error by Budget

The left panel shows 1D and the right panel shows 2D. Every line represents one beam width. Values are averaged over all selected field families, $K$ values, and $M$ values.


In [ ]:
# Close old figures, then reload and apply the shared style.
plt.close("all")
import plot_style
plot_style = importlib.reload(plot_style)
plot_style.configure_plot_style()

# Load saved records when the expensive run cell was intentionally skipped.
if "records_by_dataset" not in globals():
    records_by_dataset = {
        dataset: load_beam_width_results(
            dataset, results_root=RESULTS_ROOT, run_scope=RUN_SCOPE
        )
        for dataset in DATASETS
    }

# Create, save, and display the diagnostic figure inline.
figure = make_beam_width_plot(
    records_by_dataset,
    results_root=RESULTS_ROOT,
    run_scope=RUN_SCOPE,
)
display(Markdown("### Beam-width diagnostic figure"))
display(figure)
plt.close(figure)


## Saved files

```text
results/beam_width/
├── extended_1d_<test|full>/tables/
├── extended_2d_<test|full>/tables/
└── summary_beam_width_<test|full>/figures/
    ├── mean_error_curves_by_beam_width.png
    └── mean_error_curves_by_beam_width.pdf
